# 🏆 Model Benchmarking & Champion Evaluation
### **SDG 8: Pekerjaan Layak & Pertumbuhan Ekonomi - Hotel Booking Cancellation Risk Platform**

Notebook ini mengevaluasi dan membandingkan secara komparatif 3 filosofi machine learning:
1. **Baseline (Linear)**: Logistic Regression (Scikit-Learn Pipeline)
2. **Challenger 1 (Bagging)**: Random Forest Classifier
3. **Challenger 2 (Boosting / Champion)**: CatBoost Classifier dengan *native categorical support*

Hasil analisis mencakup perbandingan metrik, kurva ROC/PR, kalibrasi ambang batas (*threshold tuning*), visualisasi Confusion Matrix, dan peringkat kepentingan fitur (*feature importance*).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# Set style visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

BENCHMARK_PATH = Path('../../reports/benchmark/benchmark_results.json')
METADATA_PATH = Path('../../backend/app/ml/models/champion/model_metadata.json')
MODEL_PATH = Path('../../backend/app/ml/models/champion/catboost_model.cbm')
PROCESSED_DIR = Path('../../data/processed')

print("Environment & Library siap.")

--- 
## 1. Ringkasan Metrik Komparasi 3 Model

In [ ]:
with open(BENCHMARK_PATH, encoding='utf-8') as f:
    benchmark_data = json.load(f)

models_list = benchmark_data['models']
comparison_rows = []
for m in models_list:
    name = m['name']
    mtype = m['type']
    metrics = m.get('metrics_val_optimal', m.get('metrics_val'))
    comparison_rows.append({
        'Model': name,
        'Philosophy': mtype,
        'Threshold': metrics['threshold'],
        'F1-Score': metrics['f1_score'],
        'Recall': metrics['recall'],
        'Precision': metrics['precision'],
        'ROC-AUC': metrics['roc_auc'],
        'PR-AUC': metrics['pr_auc'],
        'Accuracy': metrics['accuracy'],
        'Train Time (s)': m['training_time_sec']
    })

df_comparison = pd.DataFrame(comparison_rows)
print("=== Matriks Perbandingan Model pada Validation Set ===")
df_comparison

--- 
## 2. Visualisasi Perbandingan Metrik (Bar Chart)

In [ ]:
metrics_to_plot = ['F1-Score', 'Recall', 'Precision', 'ROC-AUC', 'PR-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

colors = ['#94a3b8', '#38bdf8', '#3b82f6']
for i, row in df_comparison.iterrows():
    values = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + (i - 1) * width, values, width, label=f"{row['Model']} ({row['Philosophy']})", color=colors[i])
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.2f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score')
ax.set_title('Komparasi Metrik 3 Filosofi Model (Linear vs Bagging vs Boosting)')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontweight='bold')
ax.set_ylim(0.60, 1.02)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

--- 
## 3. Threshold Analysis & Trade-off Curve (CatBoost)

In [ ]:
th_results = benchmark_data['threshold_scan_results']
df_th = pd.DataFrame(th_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_th['threshold'], df_th['f1_score'], label='F1-Score', color='#3b82f6', linewidth=2.5)
ax.plot(df_th['threshold'], df_th['recall'], label='Recall (Cancellation)', color='#22c55e', linewidth=2, linestyle='--')
ax.plot(df_th['threshold'], df_th['precision'], label='Precision', color='#f59e0b', linewidth=2, linestyle=':')

opt_th = benchmark_data['models'][2]['optimal_threshold']
ax.axvline(opt_th, color='#ef4444', linestyle='-.', label=f'Optimal Cutoff ({opt_th})')

ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Analisis Ambang Batas (Threshold Tuning) untuk CatBoost')
ax.legend(loc='lower center')
plt.tight_layout()
plt.show()

--- 
## 4. Evaluasi Champion CatBoost pada Held-Out Test Set (17.882 Baris)

In [ ]:
with open(METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

test_metrics = metadata['metrics_test']
print(f"=== Metrik Final Test Set (Threshold = {metadata['optimal_threshold']}) ===")
for k, v in test_metrics.items():
    if k != 'confusion_matrix':
        print(f"{k.upper()}: {v}")

cm = test_metrics['confusion_matrix']
cm_matrix = np.array([[cm['tn'], cm['fp']], [cm['fn'], cm['tp']]])

fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.matshow(cm_matrix, cmap='Blues')
fig.colorbar(cax)

for (i, j), val in np.ndenumerate(cm_matrix):
    ax.text(j, i, f"{val:,}", ha='center', va='center', color='black' if val < 5000 else 'white', fontweight='bold')

ax.set_xticklabels(['', 'Pred: 0', 'Pred: 1'])
ax.set_yticklabels(['', 'Actual: 0', 'Actual: 1'])
ax.set_title('Confusion Matrix - Test Set', pad=20)
plt.tight_layout()
plt.show()

--- 
## 5. Feature Importance (Top 15 Fitur Paling Berpengaruh)

In [ ]:
model = CatBoostClassifier()
model.load_model(str(MODEL_PATH))

feature_names = metadata['features']['all_features']
importances = model.get_feature_importance()

df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(df_importance['Feature'], df_importance['Importance'], color='#3b82f6')
ax.set_xlabel('Feature Importance Score (%)')
ax.set_title('Top 15 Feature Importance - Champion CatBoost Model')
plt.tight_layout()
plt.show()